# PySpark - DataFrames

## Create a DataFrame from a CSV file

Set up up the Spark master

- "local" for local execution
- "local[*]" for local execution using all core
- "spark://spark-master:7077" to connect to the Spark master running on the docker container


In [ ]:
master = "local"
dataFolder = "/data/lab02/" if master == "spark://spark-master:7077" else "../data/"

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import column

spark = SparkSession.builder \
             .master(master) \
             .appName('flights') \
             .getOrCreate()

filename = dataFolder + 'flights.csv'
df = spark.read.option("delimiter", ",").option("header", True).option("inferSchema", "true").csv(filename)

df.printSchema()
df.show(5)


### Recast the type of a column

In [ ]:
from pyspark.sql.types import  DoubleType

def convert_column(df, column, newType):
    return df.withColumn(column, df[column].cast(newType))


df2 = convert_column(df, "dep time", DoubleType())
df2.printSchema()
df2.select("dep time").show(5)

###  Defining a custom schema

In [ ]:
from pyspark.sql.types import StringType, StructField, StructType, LongType, DoubleType, IntegerType

schema = (StructType([
					StructField("DayOfMonth", StringType(), False),
					StructField("DayOfWeek", StringType(), False),
					StructField("Carrier", StringType(), False),
					StructField("TailNum", StringType(), False),
					StructField("FlNum", IntegerType(), False),
					StructField("OrgId", LongType(), False),
					StructField("Origin", StringType(), False),
					StructField("DestId", LongType(), False),
					StructField("Dest", StringType(), False),
					StructField("ScheduledDepTime", DoubleType(), False),
					StructField("DepTime", DoubleType(), False),
					StructField("DepartureDelay", DoubleType(), False),
					StructField("ScheduledArrTime", DoubleType(), False),
					StructField("ArrTime", DoubleType(), False),
					StructField("ArrivalDelay", DoubleType(), False),
					StructField("ElapsedTime", DoubleType(), False),
					StructField("Distance", IntegerType(), False)]))

df = spark.read.schema(schema).option("delimiter", ",").option("header", True).csv(filename)

df.printSchema()
df.show(5)

## Transformations: select, add, ...

In [ ]:
df.select("Carrier").show()

In [ ]:
df.select("Origin","Dest").show()


In [ ]:
(df.selectExpr(
   "Origin","Dest",
    "(Distance > 1000) as LongDistance") # where condition
   .distinct().orderBy("Dest").show(100)
)

### Add

In [ ]:
from pyspark.sql.functions import expr

df2 = df.withColumn("FlightTime",  expr("ArrTime - DepTime")).select("Origin", "Dest", "FlightTime")
df2.show()

## Filter

In [ ]:
df.where("DepTime > 2300").show()

## Aggregations

List of functions - https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql.html#functions

Flights per Route

In [ ]:

df.groupBy("Origin", "Dest").count().show()


In [ ]:
pandas_df = df.toPandas()
pandas_df

In [ ]:
pandas_df.info()

Select first 5 columns

In [ ]:
pandas_df[0:5]

In [ ]:
result = pandas_df.groupby("Origin").agg({'Distance' : 'min'})
result

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
# plot the results
result.plot(kind='bar', title='Minimum distance');